In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")


import  pandas as pd
from src.cleaning import *

raw = pd.read_csv("../data/historical_data.csv",
                  parse_dates=["created_at", "actual_delivery_time"])


df, drop_log, store_map = clean_training_data(raw)


display(drop_log)
print("Rows before:", len(raw), "| after:", len(df))

#Sanity checks
assert  df["duration"].between(MIN_DURATION_S, MAX_DURATION_S).all()
for col in NON_NEGATIVE_COLS:
    assert (df[col].dropna() >= 0).all()

both = df[["min_item_price", "max_item_price"]].dropna()
assert (both["min_item_price"] <= both["max_item_price"]).all()
assert (df["subtotal"].dropna() > 0).all()
assert (df["total_items"] <= MAX_ITEMS).all()
assert df["store_primary_category"].notna().all()
assert len(df) + drop_log["rows_removed"].sum() == len(raw)
print("All checks passed")


,rule,rows_removed
0,missing target,7
1,duration > 10800 s,138
2,duration < 300 s,3
3,total_items > 50,8


Rows before: 197428 | after: 197272
All checks passed


In [4]:
# Impact of fix_values: NaN counts before vs after (before any rows are dropped)
stage = fix_values(add_target(raw))
cols = ["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders",
        "min_item_price", "max_item_price", "subtotal"]

impact = pd.DataFrame({
    "nan_before": raw[cols].isna().sum(),
    "nan_after": stage[cols].isna().sum(),
})
impact["newly_nan"] = impact["nan_after"] - impact["nan_before"]
display(impact)

# Effect of cleaning on the target
display(pd.DataFrame({
    "before": add_target(raw)["duration"].describe(),
    "after": df["duration"].describe(),
}))

,nan_before,nan_after,newly_nan
total_onshift_dashers,16262,16283,21
total_busy_dashers,16262,16283,21
total_outstanding_orders,16262,16306,44
min_item_price,0,813,813
max_item_price,0,800,800
subtotal,0,179,179


,before,after
count,1.974210e+05,197272.000000
mean,2.908257e+03,2852.055162
std,1.922961e+04,1082.437519
min,1.010000e+02,321.000000
25%,2.104000e+03,2104.000000
50%,2.660000e+03,2659.000000
75%,3.381000e+03,3379.000000
max,8.516859e+06,10791.000000


In [5]:
import json, os

os.makedirs("../data/processed", exist_ok=True)
df.to_parquet("../data/processed/clean.parquet", index=False)

with open("../data/processed/store_category_map.json", "w") as f:
    json.dump({str(k): v for k, v in store_map.items()}, f)

print("saved", df.shape)

saved (197272, 17)
